# Chart Out Powerball Numbers

In [11]:
import pandas as pd
import plotly.express as px
import random
import numpy as np
from sklearn.cluster import KMeans
from scipy.stats import norm

In [12]:
# Create a DataFrame
df = pd.read_csv("Powerball Numbers.csv")

print(f"{len(df):,} Rows Loaded")
df.head()

881 Rows Loaded


,Date,B1,B2,B3,B4,B5,P1
0,"Monday, July 6, 2026",17.0,44.0,63.0,66.0,67.0,4.0
1,"Saturday, July 4, 2026",17.0,38.0,46.0,50.0,69.0,20.0
2,"Wednesday, July 1, 2026",2.0,6.0,26.0,39.0,68.0,6.0
3,"Monday, June 29, 2026",10.0,14.0,41.0,53.0,59.0,3.0
4,"Saturday, June 27, 2026",3.0,16.0,28.0,30.0,59.0,11.0


In [13]:
# Melt the DataFrame to long format for plotting
df_melted = df.melt(id_vars=["Date"], value_vars=["B1", 
                                                  "B2", 
                                                  "B3", 
                                                  "B4", 
                                                  "B5", 
                                                #   "P1"
                                                  ],
                    var_name="Ball Position", value_name="Number")

# Count frequency of each number per ball position
frequency_df = df_melted.groupby(["Ball Position", "Number"]).size().reset_index(name="Frequency")

# Create a bar chart using Plotly
fig = px.bar(frequency_df, x="Number", y="Frequency", 
            #  color="Ball Position",
             title="Frequency of Powerball Numbers by Ball Position")

# Save the chart as PNG
fig.write_image("powerball_frequency_chart.png")

print(f"Saved powerball_frequency_chart.png successfully.")  

Saved powerball_frequency_chart.png successfully.


In [14]:
# Melt the DataFrame to long format for plotting
df_melted = df.melt(id_vars=["Date"], value_vars=["B1", 
                                                  "B2", 
                                                  "B3", 
                                                  "B4", 
                                                  "B5", 
                                                #   "P1"
                                                  ],
                    var_name="Ball Position", value_name="Number")

# Count frequency of each number per ball position
frequency_df = df_melted.groupby(["Ball Position", "Number"]).size().reset_index(name="Frequency")

# Create a bar chart using Plotly
fig = px.bar(frequency_df, x="Number", y="Frequency", color="Ball Position",
             title="Frequency of Powerball Numbers by Ball Position")

# Save the chart as PNG
fig.write_image("powerball_frequency_chart2.png")
print(f"Saved powerball_frequency_chart2.png successfully.")  

Saved powerball_frequency_chart2.png successfully.


In [15]:
# Melt the DataFrame to long format
df_melted = df.melt(id_vars=["Date"], value_vars=["B1", "B2", "B3", "B4", "B5", "P1"],
                    var_name="Ball Position", value_name="Number")

# Generate and save distribution chart for each ball position
ball_positions = df_melted["Ball Position"].unique()

for ball in ball_positions:
    subset = df_melted[df_melted["Ball Position"] == ball]
    freq_df = subset["Number"].value_counts().reset_index()
    freq_df.columns = ["Number", "Frequency"]
    freq_df = freq_df.sort_values("Number")

    fig = px.bar(freq_df, x="Number", y="Frequency", title=f"Distribution of {ball} Numbers",
                 labels={"Number": f"{ball} Number", "Frequency": "Frequency"})

    fig.write_image(f"{ball}_distribution_chart.png")

print(f"Saved ball_distribution_chart.png successfully.")    

Saved ball_distribution_chart.png successfully.


# Basic ML: Apply Basic Constraints, Ranges, Historical Combinations

In [16]:
# --- Analyze ranges and means ---
ball_positions = ["B1", "B2", "B3", "B4", "B5", "P1"]
range_analysis = {}
for ball in ball_positions:
    range_analysis[ball] = {
        "min": df[ball].min(),
        "max": df[ball].max(),
        "mean": round(df[ball].mean(), 2)
    }

print("Pattern Analysis for Each Ball Position:")
for ball, stats in range_analysis.items():
    print(f"{ball}: Min={stats['min']}, Max={stats['max']}, Mean={stats['mean']}")

# --- Define realistic ranges ---
ball_ranges = {
    "B1": (1, 30),
    "B2": (2, 40),
    "B3": (3, 50),
    "B4": (4, 60),
    "B5": (11, 69),
    "P1": (1, 26)
}

# --- Exclude historical combinations ---
historical_combos = set(tuple(sorted(row[:5]) + [row[5]]) for row in df[["B1", "B2", "B3", "B4", "B5", "P1"]].values)

# --- Check for sequential numbers ---
def is_sequential(combo):
    sorted_combo = sorted(combo)
    return all(sorted_combo[i+1] - sorted_combo[i] == 1 for i in range(len(sorted_combo) - 1))

# --- Weighted sampler based on historical frequency ---
def get_weighted_sampler(column, low, high):
    counts = df[column].value_counts().reindex(range(low, high+1), fill_value=1)
    weights = counts / counts.sum()
    return lambda: random.choices(population=weights.index.tolist(), weights=weights.tolist(), k=1)[0]

# --- Create samplers for each ball ---
samplers = {
    ball: get_weighted_sampler(ball, *ball_ranges[ball])
    for ball in ball_positions
}

# --- Generate predictions ---
def generate_predictions(n=5):
    predictions = []
    attempts = 0
    while len(predictions) < n and attempts < 10000:
        # ✅ Call each sampler to get a random number
        balls = sorted([samplers[f"B{i+1}"]() for i in range(5)])
        p1 = samplers["P1"]()

        combo = tuple(balls + [p1])

        # Exclude duplicates or unwanted patterns
        if tuple(sorted(combo[:5]) + [combo[5]]) in historical_combos:
            attempts += 1
            continue
        if is_sequential(combo[:5]):
            attempts += 1
            continue

        predictions.append(combo)
    return predictions

# --- Output predictions ---
print("\nPredicted Future Combinations:")
for i, combo in enumerate(generate_predictions(), 1):
    print(f"Prediction {i}: B1={combo[0]}, B2={combo[1]}, B3={combo[2]}, B4={combo[3]}, B5={combo[4]}, P1={combo[5]}")
# --- End of Script ---

Pattern Analysis for Each Ball Position:
B1: Min=1.0, Max=52.0, Mean=11.93
B2: Min=2.0, Max=59.0, Mean=23.7
B3: Min=3.0, Max=66.0, Mean=35.92
B4: Min=13.0, Max=68.0, Mean=47.36
B5: Min=22.0, Max=69.0, Mean=58.58
P1: Min=1.0, Max=26.0, Mean=13.32

Predicted Future Combinations:
Prediction 1: B1=9, B2=20, B3=32, B4=41, B5=62, P1=12
Prediction 2: B1=3, B2=30, B3=39, B4=41, B5=55, P1=11
Prediction 3: B1=12, B2=16, B3=20, B4=31, B5=69, P1=6
Prediction 4: B1=1, B2=16, B3=31, B4=46, B5=49, P1=11
Prediction 5: B1=10, B2=27, B3=27, B4=45, B5=67, P1=11


# Advanced ML: Clustering & Probability Modeling
Clustering (K-Means) / Probabilty Modeling (Normal Disctribution) / Applies Constraints

In [17]:
# Drop rows with any NaN values
df.dropna(inplace=True)
ball_positions = ["B1", "B2", "B3", "B4", "B5", "P1"]

# Clustering historical draws
X = df[ball_positions].values
kmeans = KMeans(n_clusters=2, random_state=42).fit(X)
df['Cluster'] = kmeans.labels_

# Fit normal distributions to each ball position
distributions = {}
for ball in ball_positions:
    mu, std = norm.fit(df[ball])
    distributions[ball] = (mu, std)

# Define realistic ranges
ball_ranges = {
    "B1": (1, 30),
    "B2": (2, 40),
    "B3": (3, 50),
    "B4": (4, 60),
    "B5": (11, 69),
    "P1": (1, 26)
}

# Exclude historical combinations
historical_combos = set(tuple(sorted(row[:5]) + [row[5]]) for row in df[ball_positions].values)

def is_sequential(combo):
    sorted_combo = sorted(combo)
    return all(sorted_combo[i+1] - sorted_combo[i] == 1 for i in range(len(sorted_combo) - 1))

def generate_predictions(n=5):
    predictions = []
    attempts = 0
    while len(predictions) < n and attempts < 10000:
        combo = []
        for ball in ball_positions:
            mu, std = distributions[ball]
            val = int(norm.rvs(loc=mu, scale=std))
            val = max(ball_ranges[ball][0], min(val, ball_ranges[ball][1]))
            combo.append(val)

        white_balls = sorted(combo[:5])
        powerball = combo[5]
        full_combo = tuple(white_balls + [powerball])

        if full_combo in historical_combos or is_sequential(white_balls):
            attempts += 1
            continue

        predictions.append(full_combo)
    return predictions

# Generate predictions
print("Advanced ML-Based Predicted Future Combinations:")
for i, combo in enumerate(generate_predictions(), 1):
    print(f"Prediction {i}: B1={combo[0]}, B2={combo[1]}, B3={combo[2]}, B4={combo[3]}, B5={combo[4]}, P1={combo[5]}")

Advanced ML-Based Predicted Future Combinations:
Prediction 1: B1=16, B2=22, B3=34, B4=38, B5=66, P1=1
Prediction 2: B1=11, B2=16, B3=49, B4=50, B5=59, P1=13
Prediction 3: B1=17, B2=24, B3=33, B4=37, B5=53, P1=26
Prediction 4: B1=13, B2=16, B3=32, B4=42, B5=53, P1=11
Prediction 5: B1=12, B2=18, B3=20, B4=53, B5=57, P1=22


In [18]:
# --- Load data ---
df = pd.read_csv("Powerball Numbers.csv")  # make sure filename matches
print("Columns in CSV:", df.columns.tolist())

# --- Define ball positions (adjust if your CSV uses different names) ---
ball_positions = ["B1", "B2", "B3", "B4", "B5", "P1"]

# --- Build normal distributions for each ball ---
distributions = {}
for ball in ball_positions:
    if ball in df.columns:
        mu, std = norm.fit(df[ball].dropna())
        distributions[ball] = (mu, std)
    else:
        raise KeyError(f"Column {ball} not found in CSV. Available: {df.columns.tolist()}")

# --- Define ranges for each ball ---
ball_ranges = {
    "B1": (1, 30),
    "B2": (2, 40),
    "B3": (3, 50),
    "B4": (4, 60),
    "B5": (11, 69),
    "P1": (1, 26)
}

# --- Historical combos to exclude ---
historical_combos = set(
    tuple(sorted(row[:5]) + [row[5]]) for row in df[ball_positions].values
)

# --- Helper: detect sequential combos ---
def is_sequential(combo):
    sorted_combo = sorted(combo)
    return all(sorted_combo[i+1] - sorted_combo[i] == 1 for i in range(len(sorted_combo) - 1))

# --- Helper: likelihood calculation ---
def calculate_likelihood(combo):
    likelihood = 1.0
    for i, ball in enumerate(ball_positions):
        mu, std = distributions[ball]
        prob = norm.pdf(combo[i], loc=mu, scale=std)

        if not np.isfinite(prob) or prob <= 0:
            return np.nan
        likelihood *= prob

    return round(likelihood * 100, 6)

# --- Main: generate valid combinations ---
def generate_valid_combinations(n=1000):
    results = []
    attempts = 0
    while len(results) < n and attempts < n * 20:
        combo = []
        for ball in ball_positions:
            mu, std = distributions[ball]
            val = int(norm.rvs(loc=mu, scale=std))
            val = max(ball_ranges[ball][0], min(val, ball_ranges[ball][1]))
            combo.append(val)

        white_balls = sorted(combo[:5])
        powerball = combo[5]
        full_combo = tuple(white_balls + [powerball])

        if full_combo in historical_combos or is_sequential(white_balls):
            attempts += 1
            continue

        likelihood = calculate_likelihood(full_combo)

        if np.isfinite(likelihood) and likelihood >= 50:
            results.append(full_combo + (likelihood,))
        attempts += 1
    return results

# --- Build DataFrame safely ---
combinations = generate_valid_combinations(n=1000)

# filter out bad rows
clean_combos = [row for row in combinations if all(np.isfinite(x) for x in row)]

columns = ["B1", "B2", "B3", "B4", "B5", "P1", "Likelihood (%)"]
combo_df = pd.DataFrame(clean_combos, columns=columns)

# final cleanup
combo_df = combo_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["Likelihood (%)"])

# --- Show results ---
print("\nTop 10 generated combinations (Likelihood >= 50%):")
print(combo_df.head(10))

# --- Example: search for a specific combo ---
search_combo = (4, 26, 43, 55, 63, 5)
match = combo_df[
    (combo_df["B1"] == search_combo[0]) &
    (combo_df["B2"] == search_combo[1]) &
    (combo_df["B3"] == search_combo[2]) &
    (combo_df["B4"] == search_combo[3]) &
    (combo_df["B5"] == search_combo[4]) &
    (combo_df["P1"] == search_combo[5])
]

print(f"\nSearch result for combination {search_combo}:")
print(match if not match.empty else "Combination not found.")

Columns in CSV: ['Date', 'B1', 'B2', 'B3', 'B4', 'B5', 'P1']

Top 10 generated combinations (Likelihood >= 50%):
Empty DataFrame
Columns: [B1, B2, B3, B4, B5, P1, Likelihood (%)]
Index: []

Search result for combination (4, 26, 43, 55, 63, 5):
Combination not found.
